# NMODL to BrainCell Walkthrough

This notebook shows the full inspectable path for the focused BrainCell conversion flow:

1. `.mod -> AST`
2. `AST -> RawBlocks`
3. `RawBlocks + AST -> CanonicalBlocks`
4. `CanonicalBlocks -> one_ion_hh_ohmic IR`
5. `IR + Jinja -> generated Python class`

CLI counterparts:

- `steps/inspect_ast.py`
- `steps/inspect_ir.py`
- `steps/render_preview.py`
- `examples/generate_braincell.py`

Current registered combination:

- `canonical_default -> one_ion_hh_ohmic -> braincell_one_ion_hh_ohmic`


In [ ]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "pipeline").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline import ast_to_json
from pipeline import build_canonical_blocks
from pipeline import build_one_ion_hh_ohmic_ir
from pipeline import extract_raw_blocks
from pipeline import parse_program
from pipeline import render_one_ion_hh_ohmic
from pipeline import summarize_one_ion_hh_ohmic_ir

mod_file = repo_root / "examples" / "kv.mod"
program = parse_program(mod_file)
raw_blocks = extract_raw_blocks(program)
canonical_blocks = build_canonical_blocks(program)
braincell_ir = build_one_ion_hh_ohmic_ir(canonical_blocks, mod_file, raw_blocks=raw_blocks)
rendered_python = render_one_ion_hh_ohmic(braincell_ir)

print(mod_file)


## Step 1: AST preview


In [ ]:
ast_payload = json.loads(ast_to_json(program))
print(program.get_node_type_name())
print(json.dumps(ast_payload, indent=2)[:1500])


## Step 2: Raw blocks


In [ ]:
print([name for name, values in raw_blocks.items() if values])
print(json.dumps(raw_blocks, indent=2)[:1500])


## Step 3: Canonical blocks


In [ ]:
print(json.dumps(canonical_blocks["NEURON"], indent=2))
print(json.dumps(canonical_blocks["BREAKPOINT"], indent=2))


## Step 4: BrainCell IR


In [ ]:
print(json.dumps(summarize_one_ion_hh_ohmic_ir(braincell_ir), indent=2))
print(json.dumps(braincell_ir, indent=2)[:2000])


## Step 5: Rendered Python


In [ ]:
print("\n".join(rendered_python.splitlines()[:120]))


To try a rejection case, replace `kv.mod` with `hh.mod` and rerun the notebook.
